# Healthcare Analytics for Doctor Visits
This dataset tracks demographic details, individual health conditions, available healthcare coverage, and annual doctor visits to analyze the key drivers influencing medical visit frequency.

Core Inquiry: What factors cause certain individuals to seek medical care significantly more often than others?

**About the data:** 5,190 adults from an Australian health survey (1977–78). `visits` = doctor visits in the **past 2 weeks**.

## Step 1: Load the data and check it

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
os.makedirs('charts', exist_ok=True)

df = pd.read_csv('1776250375-P2-Healthcare_Analytics_for_Doctor_Visits_1.csv')
print('Rows, columns:', df.shape)
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
df.head()

No missing values and no duplicates, so the data is clean to start with.

## Step 2: Clean and rename
- `age` is stored as years ÷ 100 (0.19 = 19 years), so we multiply by 100.
- `income` is in tens of thousands of dollars (0.55 ≈ $5,500, 1977–78 Australian dollars).
- We add `visited` = 100 if the person visited at least once, else 0. The average of this column is the **% who visited**.

In [ ]:
df = df.rename(columns={'Unnamed: 0': 'record_id'})
df['age_years'] = (df['age'] * 100).round().astype(int)
df['income_dollars'] = df['income'] * 10000
df['visited'] = (df['visits'] > 0) * 100
df[['age', 'age_years', 'income', 'income_dollars', 'visits', 'visited']].head()

## Step 3: The big picture

In [ ]:
print('% who visited at least once:', round(df['visited'].mean(), 1))
print('Average visits per person:', round(df['visits'].mean(), 2))
df['visits'].value_counts().sort_index()

In [ ]:
# A small helper so every chart looks the same
TEAL = '#0B5D63'

def bar_chart(labels, values, title, xlabel, filename, ylabel='% who visited a doctor'):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    bars = ax.bar([str(l) for l in labels], values, color=TEAL)
    ax.bar_label(bars, fmt='%.1f%%', fontsize=10)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(values) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('charts/' + filename, dpi=150)
    plt.show()

# Chart 1: how many visits did people make?
counts = df['visits'].value_counts().sort_index()
bar_chart(counts.index, counts.values / len(df) * 100, 'Most people made no doctor visits', 'Number of visits', '01_visits_distribution.png', ylabel='% of people')

**Finding:** about 80% of people made **zero** visits. Because of this, we compare the **% who visited at least once** across groups, which is easy to read.

## Step 4: Does being sick matter?

In [ ]:
def pct_visited(column):
    return df.groupby(column, observed=True)['visited'].mean()

r = pct_visited('illness')
bar_chart(r.index, r.values, 'More illnesses, more doctor visits', 'Number of illnesses (past 2 weeks)', '02_illness.png')

In [ ]:
df['reduced_days_group'] = pd.cut(df['reduced'], [-1, 0, 3, 7, 14], labels=['0', '1-3', '4-7', '8-14'])
r = pct_visited('reduced_days_group')
bar_chart(r.index, r.values, 'Days of reduced activity is the strongest signal', 'Days of reduced activity (past 2 weeks)', '03_reduced_days.png')

**Finding:** people who were more ill, or whose daily activity was cut down for more days, visited a doctor much more often.

## Step 5: Age and gender

In [ ]:
df['age_group'] = pd.cut(df['age_years'], [0, 24, 34, 44, 54, 64, 100], labels=['15-24', '25-34', '35-44', '45-54', '55-64', '65+'])
r = pct_visited('age_group')
bar_chart(r.index, r.values, 'Older people visit more', 'Age group (years)', '04_age_group.png')

r = pct_visited('gender')
bar_chart(r.index, r.values, 'Women visited more than men', 'Gender', '05_gender.png')

## Step 6: Chronic conditions and insurance
- `lchronic` = chronic condition that **limits** activity
- `nchronic` = chronic condition that does **not** limit activity
- `private` = private insurance; `freepoor` = free government cover (low income); `freerepat` = free government cover (old age, disability or veteran)

In [ ]:
r = pct_visited('lchronic')
bar_chart(['No limiting\nchronic condition', 'Limiting\nchronic condition'], r.values, 'Limiting chronic conditions double the visit rate', '', '06_chronic_limiting.png')

labels = ['Private\ninsurance', 'Free (low\nincome)', 'Free (old age /\ndisability / veteran)']
vals = [df[df[c] == 'yes']['visited'].mean() for c in ['private', 'freepoor', 'freerepat']]
bar_chart(labels, vals, '% who visited, by insurance type', '', '07_insurance.png')

In [ ]:
# Group sizes matter: the 'freepoor' group is small, so be careful with it
for c in ['private', 'freepoor', 'freerepat']:
    print(c, 'yes =', (df[c] == 'yes').sum(), 'people')

## Conclusions
1. Only about **1 in 5** people visited a doctor.
2. **Sicker** people visit most: more illnesses, more days of reduced activity, worse health score.
3. **Older** people and **women** visit more.
4. **Private insurance** made almost no difference (20% vs 20%).
5. Free government cover for old age/disability/veterans goes with more visits (32%); low-income cover goes with fewer (9%, but only 222 people).

**Be careful:** these are patterns, not proof of cause. Illness and visits are measured in the same 2 weeks, and the survey oversampled young and old people, so it is not a perfect picture of everyone.